# Expérience 5 — combinaison, puis mesure finale sur le test

Deux parties :

1. **sur la validation** : combien de places sur cinq faut-il réserver à la
   personnalisation ?
2. **sur le test** : une seule mesure, avec la meilleure configuration de chaque
   méthode. C'est le seul chiffre à citer comme résultat.

La séparation est volontaire : régler un modèle sur les données qui servent à
l'annoncer revient à l'évaluation biaisée que ce découpage corrige.

## Protocole commun

Identique dans tous les notebooks d'expérimentation, sinon les chiffres ne sont pas
comparables :

- **découpage temporel 60 / 20 / 20** sur `click_timestamp` ;
- artefacts construits sur la **seule** période d'entraînement (`models_split/`) ;
- réglage sur la **validation** ; la période de test reste intacte jusqu'à la mesure
  finale (notebook 07) ;
- lecteurs évalués : connus à l'entraînement **et** actifs pendant la période
  d'évaluation ;
- métriques : HitRate@5, Recall@5, couverture, personnalisation.

> Prérequis : `python -m src.evaluate --data-dir data/news-portal-user --out-dir models_split`

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append('..')

from src import experiments as xp
from src.recommender import Recommender

DATA = Path('..') / 'data' / 'news-portal-user'
MODELS = Path('..') / 'models_split'

train, val, test = xp.load_split(DATA)
reco = Recommender(MODELS)
users, cible = xp.eval_users(reco, val, max_users=2000)
print(f'{len(users):,} lecteurs évalués sur la période de validation')

[clicks] 1 fichier(s) vide(s) ignoré(s) : clicks_hour_100.csv


[split] entraînement 1,792,908 | validation 597,636 | test 597,637
[split] bornes temporelles : t60=1507602953792 t80=1507843212579


2,000 lecteurs évalués sur la période de validation


## 1. Coût d'exposition (validation)

Chaque place donnée au contenu est une place retirée à la popularité récente. La
question est de savoir si elle est rentable.

In [2]:
# Chaque méthode reçoit le vivier qui lui est optimal (notebooks 03 et 04) :
#   popularité -> 1 h (0,2190)   contenu -> 6 h avec profil complet (0,0405)
POOL_POPULARITE, POOL_CONTENU = 1, 6
pool_pop = xp.recent_pool(train, POOL_POPULARITE)
pool = xp.recent_pool(train, POOL_CONTENU)
popularite = xp.make_popularity(pool_pop, reco)
contenu = xp.make_content(reco, pool, last_k=None)

# ALS ré-entraîné avec sa meilleure configuration (notebook 05) : fenêtre de
# 24 h et 16 facteurs. Les artefacts de `models_split` utilisent toute la
# période et 50 facteurs, ce qui le désavantagerait ici.
als = xp.train_als_window(train, 24, factors=16)(pool, reco)

configs = {'0 place (popularité seule)': popularite}
for places in (1, 2, 3):
    configs[f'{places} place(s) au contenu'] = xp.make_mix(popularite, contenu, places)
configs['5 places (contenu seul)'] = contenu

xp.compare(configs, users, cible, n_articles=reco.n_articles)

  0%|          | 0/15 [00:00<?, ?it/s]

[als] fenêtre 24 h : 72,055 users x 6,777 items, 256,329 clics


,HitRate@5,Recall@5,couverture %,personnalisation %
configuration,,,,
0 place (popularité seule),0.2190,0.0489,0.004,5.4
1 place(s) au contenu,0.2190,0.0470,0.130,23.3
2 place(s) au contenu,0.2100,0.0417,0.194,42.4
3 place(s) au contenu,0.2005,0.0362,0.240,60.6
5 places (contenu seul),0.0405,0.0059,0.313,95.1


## 2. Complémentarité des méthodes

Si un sélecteur parfait choisissait la bonne méthode pour chaque lecteur, jusqu'où
irait-on ? Cette borne indique s'il vaut la peine de chercher une règle de
sélection.

In [3]:
strategies = {'popularité': popularite, 'contenu': contenu,
              'ALS': als}

union = 0
exclusifs = {nom: 0 for nom in strategies}
for u in users:
    trouve = {nom: len(cible[u] & set(f(u, 5))) > 0 for nom, f in strategies.items()}
    union += any(trouve.values())
    for nom, ok in trouve.items():
        if ok and sum(trouve.values()) == 1:
            exclusifs[nom] += 1

print(f'oracle (union des trois)  HitRate@5 = {union / len(users):.4f}')
print('lecteurs trouvés par une seule méthode :')
for nom, n in exclusifs.items():
    print(f'   {nom:12s} {n:4d}')

oracle (union des trois)  HitRate@5 = 0.2525
lecteurs trouvés par une seule méthode :
   popularité    362
   contenu        60
   ALS             7


## 3. Mesure finale sur le test

À n'exécuter **qu'une fois**, après avoir figé les réglages : les configurations
sont celles retenues sur la validation, seule la période d'évaluation change.

**Point de protocole essentiel.** La fenêtre de fraîcheur et l'entraînement de
l'ALS doivent être recalculés à partir de **tout ce qui précède la période de
test**, c'est-à-dire `train + validation`. Ce n'est pas une fuite : au moment de
servir une recommandation, un système en production connaît l'historique jusqu'à
l'instant présent.

Utiliser la fenêtre figée à la fin de l'entraînement donnerait un vivier vieux de
plus de deux jours — et un HitRate de **0,000** pour toutes les méthodes. Nous
l'avons mesuré : c'est la démonstration la plus nette de l'importance de la
fraîcheur dans ce projet.

In [4]:
# Historique disponible au moment du test : entraînement + validation.
historique = pd.concat([train, val], ignore_index=True)

pool_pop_test = xp.recent_pool(historique, POOL_POPULARITE)
pool_test = xp.recent_pool(historique, POOL_CONTENU)
print(f'vivier popularité : {pool_pop_test.size:,} articles | '
      f'vivier contenu : {pool_test.size:,} articles')

# Le profil des lecteurs doit lui aussi inclure la validation.
for user_id, articles in val.groupby('user_id')['click_article_id']:
    ancien = reco.user_clicks.get(int(user_id))
    nouveaux = articles.to_numpy(dtype=np.int64)
    reco.user_clicks[int(user_id)] = (nouveaux if ancien is None
                                      else np.concatenate([ancien, nouveaux]))

# Les deux modèles collaboratifs sont ré-entraînés sur `historique` : c'est ce
# dont disposerait un service en production à l'instant de répondre. Donner la
# validation à l'ALS et la refuser au SVD ferait une comparaison inégale — le
# SVD tombe alors à 0,0005 au lieu de 0,0220, par manque de données et non par
# faiblesse du modèle.
#
# Les deux variantes de note sont entraînées ici **explicitement**. Lire les
# artefacts du dossier ferait dépendre l'étiquette de la ligne de ce qui s'y
# trouve : la ligne « étoiles » a déjà affiché les chiffres de la variante
# binaire après une reconstruction des artefacts, sans qu'aucune erreur ne se
# lève.
from src.evaluate import svd_strategie

svd_negatifs = svd_strategie(reco, historique, pool_test, negatives=4)
svd_etoiles = svd_strategie(reco, historique, pool_test, negatives=0,
                            models_dir=MODELS)

popularite_test = xp.make_popularity(pool_pop_test, reco)
contenu_test = xp.make_content(reco, pool_test, last_k=None)
als_test = xp.train_als_window(historique, 24, factors=16)(pool_test, reco)

users_test, cible_test = xp.eval_users(reco, test, max_users=2000)
print(f'{len(users_test):,} lecteurs évalués sur la période de test')

MEILLEURES = {
    'popularité récente 1 h': popularite_test,
    'contenu (récents 6 h, profil complet)': contenu_test,
    'ALS (24 h, 16 facteurs)': als_test,
    'SVD Surprise (étoiles)': svd_etoiles,
    'SVD Surprise (binaire + 4 négatifs)': svd_negatifs,
    'mixte 4 pop + 1 contenu': xp.make_mix(popularite_test, contenu_test, 1),
}
final = xp.compare(MEILLEURES, users_test, cible_test, n_articles=reco.n_articles)
final

vivier popularité : 684 articles | vivier contenu : 2,101 articles


[notes] 2,359,791 positifs + 9,419,560 négatifs = 11,779,351 exemples


[notes] 2,075,824 couples notés à partir des étoiles — 1★:215,155 · 2★:209,080 · 3★:342,147 · 4★:946,799 · 5★:362,643


  0%|          | 0/15 [00:00<?, ?it/s]

[als] fenêtre 24 h : 39,433 users x 4,454 items, 125,753 clics


2,000 lecteurs évalués sur la période de test


,HitRate@5,Recall@5,couverture %,personnalisation %
configuration,,,,
popularité récente 1 h,0.2525,0.0555,0.003,4.0
"contenu (récents 6 h, profil complet)",0.0200,0.0034,0.268,96.1
"ALS (24 h, 16 facteurs)",0.0415,0.0047,0.015,94.9
SVD Surprise (étoiles),0.0000,0.0000,0.005,11.8
SVD Surprise (binaire + 4 négatifs),0.0220,0.0029,0.010,55.9
mixte 4 pop + 1 contenu,0.2500,0.0539,0.116,21.5


## Conclusion à retenir pour la présentation

Résultats mesurés sur la période de **test**, chaque méthode dans sa meilleure
configuration réglée sur la validation :

| Configuration | HitRate@5 | Couverture | Personnalisation |
|---|---|---|---|
| popularité récente 1 h | **0,2525** | 0,003 % | 4,0 % |
| mixte 4 popularité + 1 contenu | **0,2500** | 0,116 % | 21,5 % |
| ALS (24 h, 16 facteurs) | 0,0415 | 0,015 % | 94,9 % |
| SVD Surprise (binaire + 4 négatifs) | 0,0220 | 0,010 % | 55,9 % |
| contenu (récents 6 h, profil complet) | 0,0200 | **0,268 %** | **96,1 %** |
| SVD Surprise (étoiles) | 0,0000 | 0,005 % | 11,8 % |

Ce sont les chiffres de `models/baseline_metrics.json`, la référence dont se sert
la porte de qualité de la CI. Une seule commande les reproduit :

```
python -m src.evaluate --split test --skip-build --json models/baseline_metrics.json
```

1. **La fraîcheur pèse plus que le modèle.** La popularité comptée sur une heure
   atteint 0,2525 ; la même popularité comptée sur tout l'historique, 0,0060 — un
   **facteur 42** sans changer une ligne d'algorithme. Le relevé complet des huit
   fenêtres est dans `models/freshness_sweep.json`
   (`python scripts/sweep_fraicheur.py`).
2. **Une place sur cinq accordée à la personnalisation est gratuite.** 0,2500
   contre 0,2525, soit 1 % d'écart sur 2 000 lecteurs, et cela achète une
   couverture **38 fois** plus large. C'est la recommandation produit défendable,
   à la place d'un `alpha = 0,5` choisi par défaut.
3. **Toutes les méthodes personnalisées sont un ordre de grandeur en dessous** de
   la popularité récente en précision (0,020–0,042 contre 0,2525).
4. **Leur valeur est ailleurs** : le contenu expose 0,268 % du catalogue contre
   0,003 % pour la popularité, avec une personnalisation de 96 %. Un éditeur qui
   ne veut pas réduire son audience à onze articles y a un intérêt direct — que le
   HitRate ne mesure pas.
5. **La formulation compte plus que la bibliothèque** : le même SVD Surprise passe
   de 0,0000 (note = étoiles de l'article) à 0,0220 (notes binaires + négatifs).

### Limites de ce protocole — à énoncer

- **Un réglage n'a pas résisté au test** : le SVD à 4 négatifs donnait 0,0815 sur
  la validation et 0,0220 sur le test, soit 3,7 fois moins. C'est précisément ce
  que la séparation validation / test sert à détecter.
- **Les optima partiels ne se combinent pas.** Les deux réglages de l'ALS avaient
  été balayés séparément, et juxtaposer leurs gagnants (72 h + 16 facteurs) donne
  la **pire** des quatre combinaisons — 0,0110 contre 0,0345 pour 24 h + 16
  facteurs. Voir le balayage croisé du notebook 05.
- **La fenêtre de fraîcheur doit être recalculée à l'instant de la requête.**
  Gelée à la fin de l'entraînement, elle donne 0,0000 pour toutes les méthodes —
  mesuré, puis corrigé en utilisant `train + validation` comme historique du test.
- **Les deux collaboratifs doivent être ré-entraînés sur cet historique.** L'ALS
  l'était, le SVD non : il arrivait d'artefacts construits sur la seule période
  d'entraînement et tombait à 0,0005, par manque de données et non par faiblesse
  du modèle. Corrigé — c'est la troisième règle de protocole de `src/evaluate.py`.
- **Le cold start n'est pas évalué ici** : seuls les lecteurs connus à
  l'entraînement sont mesurés. Un nouveau lecteur reçoit la popularité récente, ce
  qui est précisément la méthode la plus précise de ce tableau.

### Ce que ce protocole a déjà fait corriger

1. **Fenêtre glissante** : `prepare_model` produit `popular_recent.npy` (1 h) et
   `candidates_recent.npy` (6 h), et la Function les relit à chaque appel par
   *blob input binding* — mise à jour horaire sans redémarrage.
2. **Vivier de candidats restreint aux articles récents** dans `Recommender`,
   activé par défaut et débrayable dans l'interface (`fresh_only`).
3. **Une place réservée au contenu** dans le top-5 : c'est la stratégie `mix`,
   celle qui est servie en production.

### Ce qui reste à construire

- recalcul **continu** de la fenêtre à partir d'un flux de clics, plutôt que par
  le traitement hors-ligne ;
- index de similarité (ANN) pour ne plus scorer les 364 047 articles à chaque
  appel ;
- évaluation du cold start comme mesure à part entière.